In [0]:
customers_df = spark.table("bronze_customers")
products_df = spark.table("bronze_products")
orders_df = spark.table("bronze_orders")

In [0]:
customers_df.show(5)
products_df.show(5)
orders_df.show(5)

In [0]:
customers_df = customers_df.dropDuplicates().na.drop()
orders_df = orders_df.dropDuplicates().na.drop()
products_df = products_df.dropDuplicates().na.drop()

In [0]:
silver_df =(
    orders_df.alias("o").join(
        customers_df.alias("c"),
        on="customer_id",
        how="inner"
    ).join(
        products_df.alias("p"),
        on="product_id",
        how="inner"
    )   
)

In [0]:
silver_df.printSchema()
silver_df.show(5, truncate=False)

In [0]:
from pyspark.sql.functions import col

silver_df = silver_df.withColumn("revenue", col("quantity") * col("unit_price"))

In [0]:
silver_df = silver_df.select(
    "order_id",
    "customer_id",
    "customer_name",
    "region",
    "product_id",
    "product_name",
    "category",
    "quantity",
    "unit_price",
    "revenue",
    "order_date",
    "status",
    "signup_date"   
)


In [0]:
silver_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_enriched_orders")